<a href="https://colab.research.google.com/github/Koquda/intro_ai_ml/blob/main/midterm_trauma_word2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Midterm: Word2Vec Literature Search for Orthopaedic Trauma Surgeons (PySpark)

**ROI idea:** trauma surgeons lose clinical time digging through PubMed for evidence
on a specific injury pattern. Keyword search misses papers that describe the same problem in different words
(*"failure of bone healing"* vs *"nonunion"*). This notebook trains a Word2Vec model on about 6,000 trauma abstracts and
ranks articles by **meaning**, returning the 50 most relevant titles for any free-text clinical question.
This replaces my original "ROI for AI" discussion idea (AI screening of government contracts) with the professor's
approval; the PDF explains the change.

| Step | Rubric item | Section |
|---|---|---|
| 1 | Find and load own dataset | §1 |
| 2 | Remove stopwords and tokenize article bodies | §2 |
| 3 | Train Word2Vec to rank documents for a query | §3 |
| 4 | Test and evaluate search queries, 50 matching articles | §4 |
| + | Train a model to predict: topic classifier on the Word2Vec vectors | §5 |

**Data:** `data/trauma_abstracts.csv` (file: `trauma_abstracts_data.zip`, submitted with this notebook), built by
`fetch_pubmed.py` from the official NCBI E-utilities API. It contains English abstracts from 2010 to 2025, retrieved
with eight trauma topic queries.
**Data link (source):** [the same eight queries on PubMed](https://pubmed.ncbi.nlm.nih.gov/?term=%28%28hip+fracture%5Btiab%5D%29+OR+%28tibial+fracture%5Btiab%5D+AND+intramedullary%5Btiab%5D%29+OR+%28nonunion%5Btiab%5D+AND+fracture%5Btiab%5D%29+OR+%28distal+radius+fracture%5Btiab%5D%29+OR+%28ankle+fracture%5Btiab%5D%29+OR+%28%28pelvic+fracture%5Btiab%5D+OR+acetabular+fracture%5Btiab%5D%29%29+OR+%28%28fracture-related+infection%5Btiab%5D+OR+open+fracture%5Btiab%5D%29%29+OR+%28polytrauma%5Btiab%5D+AND+%28fixation%5Btiab%5D+OR+damage+control%5Btiab%5D%29%29%29+AND+hasabstract+AND+english%5Bla%5D+AND+2010%3A2025%5Bdp%5D). PubMed adds new papers over time,
so the live search returns slightly more articles than the frozen CSV used here.

**How the code is organised:** every tunable setting is in the configuration cell (§0.3). Each processing step is a
small, typed function with a docstring that receives what it needs as arguments. The search engine (§4) is a class
that owns its model and article vectors.

## 0. Setup

### 0.1 Install dependencies

The cell detects where it is running and installs only what is missing:

| | Google Colab | Local (Jupyter / VS Code) |
|---|---|---|
| Java | Colab ships Java 11, but Spark 4 needs **Java 17+**, so OpenJDK 17 is installed | Install **Java 17 or 21** yourself and set `JAVA_HOME` |
| Python packages | `pyspark==4.0.1`, `numpy` | same |
| Data | Upload `trauma_abstracts_data.zip` (or the CSV) to the *Files* panel, or wait for the upload dialog in §1 | Keep `trauma_abstracts_data.zip` next to this notebook; §1 unzips it |

In [1]:
import os   # environment variables (JAVA_HOME) and file checks
import sys  # list of loaded modules, used to detect Colab

# 1. Detect the environment: Google Colab always loads the "google.colab" module when the runtime starts
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # 2. Spark 4 requires Java 17+, but Colab only has Java 11: install OpenJDK 17 quietly (-qq) without prompts (-y)
    !apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
    # 3. Tell Spark where the new Java lives
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

# 4. Install the Python packages into this kernel (both environments); -q hides the installation log
%pip install -q pyspark==4.0.1 numpy
print("Environment:", "Google Colab" if IN_COLAB else "local")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Environment: Google Colab


### 0.2 Imports

In [2]:
# Python standard library
import glob     # find the data file wherever it was uploaded
import os       # file checks
import zipfile  # unpack the submitted data archive
from collections.abc import Callable  # type of a function passed as an argument (the preprocess step)
from dataclasses import dataclass     # compact classes that only hold data (DocumentIndex, EvalQuery)

# Numerical computing: the similarity maths on the collected article vectors
import numpy as np
import numpy.typing as npt  # array types for the annotations

# Spark: DataFrames, column functions and the MLlib models
import pyspark.sql.functions as F  # column functions (filter, lower, split, avg, ...), same alias as the course
from pyspark.ml.classification import LogisticRegression, LogisticRegressionModel  # topic classifier (§5)
from pyspark.ml.evaluation import MulticlassClassificationEvaluator               # classifier metrics (§5)
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, StringIndexer, Word2Vec, Word2VecModel
from pyspark.sql import DataFrame, SparkSession

# Type aliases used in the function signatures below
FloatArray = npt.NDArray[np.float64]
BoolArray = npt.NDArray[np.bool_]

### 0.3 Configuration
Every setting that changes the results lives here, so an experiment can be reproduced from this one cell.

In [3]:
# Data
DATA_PATH = "data/trauma_abstracts.csv"            # where the CSV is expected, relative to this notebook
DATA_ARCHIVE_PATTERN = "trauma_abstracts_data*.zip"  # the submitted zip; * also matches browser renames like " (1)"

# Tokenization
TOKEN_PATTERN = r"[a-z][a-z0-9]*"  # a token must start with a letter, so numbers, p-values and percentages are dropped
MIN_TOKEN_LENGTH = 2               # drops single letters left over from abbreviations ("p", "n")
ABSTRACT_FILLER = [                # boilerplate found in almost every abstract; it says nothing about the topic
    "study", "studies", "patients", "patient", "results", "result", "conclusion", "conclusions",
    "methods", "method", "background", "objective", "objectives", "purpose", "aim", "aims",
    "using", "used", "use", "also", "however", "among", "group", "groups", "total", "included",
    "significant", "significantly", "ci", "vs", "versus", "years", "year", "data", "level", "evidence",
]

# Word2Vec
VECTOR_SIZE = 100  # dimensions per word vector (same as the course notebook)
MIN_COUNT = 5      # ignore words seen fewer than 5 times (typos, one-off terms)
WINDOW_SIZE = 5    # context = 5 words on each side of the target word
MAX_ITER = 5       # passes over the corpus; one pass is too few for about 6k abstracts
SEED = 42          # fixed random start, so every run gives the same model

# Search
TOP_K = 50         # articles returned per query (assignment requirement)

# Topic prediction (§5)
TEST_FRACTION = 0.2  # share of articles held out to test the classifier

### 0.4 Spark session

In [4]:
def create_spark_session(app_name: str) -> SparkSession:
    '''Start a local Spark session on all CPU cores, with quiet logs so the saved outputs stay readable.'''
    spark = (
        SparkSession.builder.master("local[*]")               # run on this machine, using every CPU core
        .appName(app_name)                                     # name shown in the Spark UI and logs
        .config("spark.driver.memory", "4g")                   # room for the collected article vectors
        .config("spark.ui.showConsoleProgress", "false")       # no progress bars in the notebook output
        .getOrCreate()                                         # reuse the session if one is already running
    )
    spark.sparkContext.setLogLevel("ERROR")  # only show real errors, not INFO/WARN messages
    return spark


# Single entry point to Spark, used by every cell below
spark = create_spark_session("trauma-word2vec")
print("Using Apache Spark Version", spark.version)

Using Apache Spark Version 4.0.1


## 1. Load the dataset (Crunchbase-style CSV)

The CSV follows the same shape as `cb_odm_092419.csv`: one row per entity with an id, a name/title and a text field.
Here the text field is the **abstract body**. Extra columns are `journal`, `year`, `url`, PubMed `mesh` subject tags
(used only for evaluation in §4) and `source_topic`, the query that retrieved the article.

In [5]:
def load_articles(spark: SparkSession, path: str) -> DataFrame:
    '''
    Read the abstracts CSV into a Spark DataFrame, as the course notebook does with the Crunchbase file.
    multiLine and escape are needed because abstracts contain commas and quoted text.
    '''
    return spark.read.options(
        header=True,       # first row holds the column names
        inferSchema=True,  # detect types automatically (pmid and year become integers)
        multiLine=True,    # a quoted field may contain line breaks
        escape='"',        # a quote inside a quoted field is written as "" (standard CSV)
    ).csv(path)


def find_file(pattern: str) -> str | None:
    '''First file matching the pattern in this folder or any subfolder, or None if there is none.'''
    matches = sorted(glob.glob(f"**/{pattern}", recursive=True))
    return matches[0] if matches else None


def locate_csv(path: str) -> str | None:
    '''The CSV at its expected path, or the same file name uploaded anywhere else in this folder.'''
    return path if os.path.exists(path) else find_file(os.path.basename(path))


def unpack_archive() -> bool:
    '''Unzip the submitted archive into this folder (it creates data/trauma_abstracts.csv). False if not found.'''
    archive = find_file(DATA_ARCHIVE_PATTERN)
    if archive is None:
        return False
    with zipfile.ZipFile(archive) as zf:
        zf.extractall()
    return True


def ensure_dataset(path: str) -> str:
    '''
    Return the path of the dataset CSV, preparing it first if needed:
    1. the CSV is already there (unzipped earlier, or uploaded on its own);
    2. otherwise the submitted zip is unpacked;
    3. in Colab, if neither file is present, an upload dialog opens and steps 1-2 are tried again.
    '''
    if csv_path := locate_csv(path):
        return csv_path
    if unpack_archive():
        return path
    if IN_COLAB:
        from google.colab import files  # only exists inside Colab

        print("Dataset not found: please upload trauma_abstracts_data.zip (or trauma_abstracts.csv)")
        files.upload()  # saves the chosen file in the current folder (/content)
        if csv_path := locate_csv(path):
            return csv_path
        if unpack_archive():
            return path
    raise FileNotFoundError(f"{path} not found: put trauma_abstracts_data.zip next to this notebook")


# Make sure the data file exists (unzipping or asking for an upload if needed), then load it
data_file = ensure_dataset(DATA_PATH)
print("dataset file:", data_file)
raw = load_articles(spark, data_file)

# 1. Column names and types: pmid (id), title, body (abstract), journal, year, mesh, source_topic, url
raw.printSchema()

# 2. Number of articles in the file
print(f"rows loaded: {raw.count():,}")

dataset file: data/trauma_abstracts.csv
root
 |-- pmid: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- body: string (nullable = true)
 |-- journal: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- mesh: string (nullable = true)
 |-- source_topic: string (nullable = true)
 |-- url: string (nullable = true)

rows loaded: 6,220


In [6]:
# 1. Preview 5 articles (long titles cut at 60 characters) to confirm the text fields loaded correctly
raw.select("pmid", "title", "journal", "year").show(5, truncate=60)

# 2. Articles per PubMed topic query, largest first: checks that no single topic dominates the dataset
raw.groupBy("source_topic").count().orderBy(F.desc("count")).show()

+--------+------------------------------------------------------------+------------------------------------------------------+----+
|    pmid|                                                       title|                                               journal|year|
+--------+------------------------------------------------------------+------------------------------------------------------+----+
|33522384|Physical Therapy Management of Older Adults With Hip Frac...|The Journal of orthopaedic and sports physical therapy|2021|
|39341619|                              The Patient with Hip Fracture.|                  The Medical clinics of North America|2024|
|29699731|              Epidemiology and social costs of hip fracture.|                                                Injury|2018|
|30063657|     Pericapsular Nerve Group (PENG) Block for Hip Fracture.|                 Regional anesthesia and pain medicine|2018|
|33289066|Guideline for the management of hip fractures 2020: Guide...|     

### 1.1 Clean: drop empty bodies and duplicate titles, build `inputText`

In [7]:
def drop_empty_bodies(df: DataFrame) -> DataFrame:
    '''Remove articles without an abstract: there is nothing to tokenize, so they could never be found.'''
    # Keep a row only if body exists (not null) AND is not just spaces
    return df.filter(F.col("body").isNotNull() & (F.trim("body") != ""))


def drop_duplicate_titles(df: DataFrame) -> DataFrame:
    '''
    Keep one article per title (the same paper indexed twice, errata, reprints) so the top 50 has no repeats.
    Titles are lowercased and trimmed first, so small formatting differences still count as duplicates.
    '''
    return (
        df.withColumn("title_key", F.lower(F.trim("title")))  # temporary normalised copy of the title
        .dropDuplicates(["title_key"])                         # keep the first article for each normalised title
        .drop("title_key")                                     # remove the helper column again
    )


def add_input_text(df: DataFrame) -> DataFrame:
    '''
    Build the inputText column the NLP pipeline reads (same name as in the course notebook).
    The title goes in front of the body because it is the densest summary of what the article is about.
    '''
    # concat_ws joins the columns with a space and skips nulls
    return df.withColumn("inputText", F.concat_ws(" ", "title", "body"))


# Apply the three cleaning steps in order. cache() keeps the result in memory because every later step reuses it.
non_empty = drop_empty_bodies(raw)
articles = add_input_text(drop_duplicate_titles(non_empty)).cache()

# Log how many rows each step removed, then preview the text that goes into the NLP pipeline
print(f"loaded {raw.count():,} -> with abstract {non_empty.count():,} -> unique titles {articles.count():,}")
articles.select("pmid", "inputText").show(3, truncate=110)

loaded 6,220 -> with abstract 6,218 -> unique titles 6,213
+--------+--------------------------------------------------------------------------------------------------------------+
|    pmid|                                                                                                     inputText|
+--------+--------------------------------------------------------------------------------------------------------------+
|28601513|A Biomechanical Comparison of Distal Fixation for Bridge Plating in a Distal Radius Fracture Model. To comp...|
|34055585|Acute syndesmotic injuries in ankle fractures: From diagnosis to treatment and current concepts. A stable a...|
|33878041|Alternative Use of the Ilizarov Apparatus Set in Case of Complications During Intramedullary Nail Removal: ...|
+--------+--------------------------------------------------------------------------------------------------------------+
only showing top 3 rows


## 2. Tokenize and remove stopwords

* **RegexTokenizer** lowercases the text and keeps tokens that start with a letter and are at least 2 characters long.
  This drops p-values, sample sizes and percentages, which carry no topical meaning.
* **StopWordsRemover** uses Spark's default English list plus the abstract boilerplate from the configuration
  ("study", "patients", "results", ...). These words appear in almost every abstract and would make all documents look alike.

Both stages are wrapped in one `preprocess` function. It is applied to the articles **and** to every search query, so
the two are always cleaned the same way.

In [8]:
# 1. RegexTokenizer (same function as the course notebook): inputText -> tokens
regextok = RegexTokenizer(
    inputCol="inputText", outputCol="tokens",
    gaps=False,                       # the pattern describes the tokens to KEEP, not the separators between them
    pattern=TOKEN_PATTERN,            # words that start with a letter (see configuration)
    minTokenLength=MIN_TOKEN_LENGTH,  # drop tokens shorter than 2 characters
    toLowercase=True,                 # "Fracture" and "fracture" become the same token
)

# 2. StopWordsRemover (same function as the course notebook): tokens -> tokens_clean
stopwrmv = StopWordsRemover(
    inputCol="tokens", outputCol="tokens_clean",
    # Spark's built-in English list ("the", "and", "of", ...) plus the abstract boilerplate from the configuration
    stopWords=StopWordsRemover.loadDefaultStopWords("english") + ABSTRACT_FILLER,
)


def preprocess(df: DataFrame) -> DataFrame:
    '''inputText -> tokens -> tokens_clean. The single cleaning path shared by articles and queries.'''
    return stopwrmv.transform(regextok.transform(df))  # tokenize first, then remove stopwords


# 3. Clean every article. Cached because Word2Vec training and the evaluation both reuse it.
corpus = preprocess(articles).cache()

In [9]:
# Before / after on one article: raw text, all tokens, then tokens with stopwords removed
example = corpus.select("inputText", "tokens", "tokens_clean").first()  # first article of the corpus
print("TEXT  :", example.inputText[:300], "...")   # first 300 characters of the raw text
print("TOKENS:", example.tokens[:30])              # first 30 tokens after RegexTokenizer
print("CLEAN :", example.tokens_clean[:30])        # first 30 tokens after StopWordsRemover ("of", "the", "using" are gone)

TEXT  : A Biomechanical Comparison of Distal Fixation for Bridge Plating in a Distal Radius Fracture Model. To compare the biomechanical properties of second versus third metacarpal distal fixation when using a radiocarpal spanning distraction plate in an unstable distal radius fracture model. Biomechanical ...
TOKENS: ['biomechanical', 'comparison', 'of', 'distal', 'fixation', 'for', 'bridge', 'plating', 'in', 'distal', 'radius', 'fracture', 'model', 'to', 'compare', 'the', 'biomechanical', 'properties', 'of', 'second', 'versus', 'third', 'metacarpal', 'distal', 'fixation', 'when', 'using', 'radiocarpal', 'spanning', 'distraction']
CLEAN : ['biomechanical', 'comparison', 'distal', 'fixation', 'bridge', 'plating', 'distal', 'radius', 'fracture', 'model', 'compare', 'biomechanical', 'properties', 'second', 'third', 'metacarpal', 'distal', 'fixation', 'radiocarpal', 'spanning', 'distraction', 'plate', 'unstable', 'distal', 'radius', 'fracture', 'model', 'biomechanical', 'evaluation', 'ra

In [10]:
# 1. Tokens per article before and after stopword removal
corpus.select(
    F.avg(F.size("tokens")).alias("avg_tokens"),              # average tokens per article after tokenizing
    F.avg(F.size("tokens_clean")).alias("avg_tokens_clean"),  # average after stopword removal
    F.min(F.size("tokens_clean")).alias("min_clean"),         # shortest cleaned article (checks none is empty)
    F.max(F.size("tokens_clean")).alias("max_clean"),         # longest cleaned article
).show()

# 2. Most frequent remaining tokens (explode = one row per token). They should read like a trauma vocabulary.
token_counts = corpus.select(F.explode("tokens_clean").alias("token")).groupBy("token").count()
print(f"distinct tokens: {token_counts.count():,}")
token_counts.orderBy(F.desc("count")).show(20)

+------------------+------------------+---------+---------+
|        avg_tokens|  avg_tokens_clean|min_clean|max_clean|
+------------------+------------------+---------+---------+
|206.85514244326413|126.50474810880412|        7|      463|
+------------------+------------------+---------+---------+

distinct tokens: 21,410
+----------+-----+
|     token|count|
+----------+-----+
|  fracture|21165|
| fractures|11594|
|  fixation| 6132|
| treatment| 5471|
|       hip| 5312|
|    distal| 4987|
|  nonunion| 4916|
|     ankle| 4798|
|      open| 4152|
|   surgery| 4001|
|    injury| 3757|
|      bone| 3696|
|  surgical| 3579|
|    radius| 3397|
|      risk| 3289|
|associated| 2946|
|  outcomes| 2939|
| reduction| 2889|
|    pelvic| 2864|
|      case| 2859|
+----------+-----+
only showing top 20 rows


## 3. Train the Word2Vec model

Word2Vec is a **predictive model**. In skip-gram training, each word is used to *predict* the words around it
(window = 5). The network weights it learns become 100-dimensional word vectors. Words used in similar contexts,
such as *nonunion* and *pseudarthrosis*, end up close together. The fitted model then *predicts* a document's position in
that space by averaging its word vectors. This is what lets the search below match on meaning instead of exact words.

In [11]:
def train_word2vec(corpus: DataFrame) -> Word2VecModel:
    '''Fit Word2Vec on the cleaned tokens using the settings from the configuration cell.'''
    word2vec = Word2Vec(
        vectorSize=VECTOR_SIZE,  # 100 numbers per word
        minCount=MIN_COUNT,      # ignore words seen fewer than 5 times
        windowSize=WINDOW_SIZE,  # 5 context words on each side
        maxIter=MAX_ITER,        # 5 passes over the corpus
        seed=SEED,               # reproducible training
        inputCol="tokens_clean",  # the output column of step 2 (StopWordsRemover)
        outputCol="wordvectors",  # where transform() writes each document's averaged vector
    )
    return word2vec.fit(corpus)  # training: learn one vector per vocabulary word


model = train_word2vec(corpus)

# getVectors() returns one row per learned word, so its count is the vocabulary size
print(f"vocabulary size: {model.getVectors().count():,}")

vocabulary size: 9,136


### 3.1 Sanity check: nearest words

In [12]:
# If the model learned meaning, the closest words to each clinical term should be clinically related
for word in ["nonunion", "infection", "nail", "wrist"]:
    print(f"--- {word}")
    model.findSynonyms(word, 8).show(truncate=False)  # 8 nearest words by cosine similarity

--- nonunion
+---------------+------------------+
|word           |similarity        |
+---------------+------------------+
|nonunions      |0.7899386286735535|
|recalcitrant   |0.6363341808319092|
|atrophic       |0.6064475774765015|
|hypertrophic   |0.5645955204963684|
|oligotrophic   |0.5643643140792847|
|union          |0.5608101487159729|
|aseptic        |0.5289517045021057|
|pseudoarthrosis|0.5177099704742432|
+---------------+------------------+

--- infection
+-------------+------------------+
|word         |similarity        |
+-------------+------------------+
|infections   |0.8648083806037903|
|fri          |0.6953636407852173|
|osteomyelitis|0.65380859375     |
|fris         |0.6196389198303223|
|pti          |0.5912140607833862|
|lgi          |0.5829160213470459|
|swc          |0.5685686469078064|
|infecting    |0.567726731300354 |
+-------------+------------------+

--- nail
+--------------+------------------+
|word          |similarity        |
+--------------+----------

## 4. Search: top 50 articles for a query

`SemanticSearch` follows the course notebook's steps:

1. **Index:** each article becomes the average of its word vectors (`model.transform`). All 6,213 vectors are collected
   once into a NumPy matrix. The course notebook kept only its first 30,000 rows; here every article is searchable.
2. **Query:** the query goes through the **same** `preprocess` function and model.
3. **Rank:** cosine similarity between the query and every article, with the same formula as the course's `cossim()`,
   computed for all articles in one matrix operation.

**Zero-vector guard:** an article with no in-vocabulary words has an all-zero vector. Its similarity is set to 0 instead
of dividing by zero. A query with no known words raises a clear error instead of returning meaningless results.

In [13]:
@dataclass(frozen=True)
class DocumentIndex:
    '''Article vectors held in memory for fast search. Row i of every array describes the same article.'''

    pmids: npt.NDArray[np.int64]
    titles: npt.NDArray[np.str_]
    vectors: FloatArray  # shape (n_articles, VECTOR_SIZE)
    norms: FloatArray    # length of each article vector, computed once and reused for every query

    @classmethod
    def build(cls, model: Word2VecModel, corpus: DataFrame) -> "DocumentIndex":
        '''Average each article's word vectors with the model and collect them into NumPy arrays.'''
        # 1. transform() adds the averaged vector per article; collect() brings the 6,213 rows to the driver (~5 MB)
        rows = model.transform(corpus).select("pmid", "title", "wordvectors").collect()
        # 2. Stack the Spark vectors into one NumPy matrix: one row per article, one column per dimension
        vectors = np.array([row.wordvectors.toArray() for row in rows])
        # 3. Keep ids and titles in the same order as the matrix rows, and precompute each row's length
        return cls(
            pmids=np.array([row.pmid for row in rows]),
            titles=np.array([row.title for row in rows]),
            vectors=vectors,
            norms=np.linalg.norm(vectors, axis=1),
        )


class SemanticSearch:
    '''Ranks articles by cosine similarity between their averaged Word2Vec vector and the query's.'''

    def __init__(self, spark: SparkSession, model: Word2VecModel,
                 preprocess: Callable[[DataFrame], DataFrame], corpus: DataFrame) -> None:
        self.spark = spark            # used to wrap queries and results in DataFrames
        self.model = model            # trained Word2Vec model
        self.preprocess = preprocess  # the same cleaning function used for the articles (§2)
        self.index = DocumentIndex.build(model, corpus)  # article vectors, built once at start-up

    def _query_df(self, query: str) -> DataFrame:
        '''Wrap the query in a one-row DataFrame and clean it exactly like the articles.'''
        return self.preprocess(self.spark.createDataFrame([(query,)], ["inputText"]))

    def tokens(self, query: str) -> list[str]:
        '''The query words the model actually sees after tokenization and stopword removal.'''
        return self._query_df(query).first().tokens_clean

    def embed(self, query: str) -> FloatArray:
        '''Average the Word2Vec vectors of the query words into one vector.'''
        return self.model.transform(self._query_df(query)).first().wordvectors.toArray()

    def similarities(self, query: str) -> FloatArray:
        '''Cosine similarity = dot(article, query) / (|article| * |query|), for every article at once.'''
        # 1. Query vector and its length. Length 0 means none of the query words are in the vocabulary,
        #    so cosine similarity is undefined: stop with a clear message instead of returning noise.
        query_vec = self.embed(query)
        query_norm = np.linalg.norm(query_vec)
        if query_norm == 0:
            raise ValueError(f"No word of {query!r} is in the Word2Vec vocabulary")
        # 2. vectors @ query_vec = the dot product with every article in one operation (same maths as cossim()).
        #    Dividing by the lengths turns it into cosine similarity; zero-vector articles keep a score of 0.
        scores = np.zeros(len(self.index.pmids))
        np.divide(self.index.vectors @ query_vec, self.index.norms * query_norm,
                  out=scores, where=self.index.norms > 0)
        return scores

    def search(self, query: str, k: int = TOP_K, pool: BoolArray | None = None) -> DataFrame:
        '''
        Return the k most similar articles as a Spark DataFrame (rank, pmid, title, similarity).
        pool is an optional True/False mask that restricts the search to some articles (used in §4.3).
        Titles are unique after §1.1, so the k results are k different articles.
        '''
        scores = self.similarities(query)
        if pool is not None:
            scores = np.where(pool, scores, -np.inf)  # articles outside the pool can never reach the top k
        # argsort on the negated scores = positions sorted from highest to lowest similarity; keep the first k
        top = np.argsort(-scores)[:k]
        # One result row per article: rank (1..k), PubMed id, title, cosine similarity
        rows = [(rank, int(self.index.pmids[i]), str(self.index.titles[i]), float(scores[i]))
                for rank, i in enumerate(top, start=1)]
        return self.spark.createDataFrame(rows, ["rank", "pmid", "title", "similarity"])


# Build the engine once (this computes all article vectors), then check for articles with no known words
engine = SemanticSearch(spark, model, preprocess, corpus)
zero_vectors = int((engine.index.norms == 0).sum())
print(f"indexed articles: {len(engine.index.pmids):,} | vector size: {engine.index.vectors.shape[1]} | "
      f"zero-vector articles: {zero_vectors}")

indexed articles: 6,213 | vector size: 100 | zero-vector articles: 0


### 4.1 Sample search query: top 50 matching titles

In [14]:
SEARCH_QUERY = "hip fracture elderly mortality surgery delay"  # any combination of words works

# 1. What the model actually receives after cleaning
print("query tokens:", engine.tokens(SEARCH_QUERY))

# 2. The 50 best-matching titles, highest similarity first (titles cut at 100 characters for readability)
engine.search(SEARCH_QUERY).show(TOP_K, truncate=100)

query tokens: ['hip', 'fracture', 'elderly', 'mortality', 'surgery', 'delay']
+----+--------+----------------------------------------------------------------------------------------------------+------------------+
|rank|    pmid|                                                                                               title|        similarity|
+----+--------+----------------------------------------------------------------------------------------------------+------------------+
|   1|30514642|Unlike Native Hip Fractures, Delay to Periprosthetic Hip Fracture Stabilization Does Not Signific...|0.8179429727008847|
|   2|34688941|                      Impact of bisphosphonates and comorbidities on initial hip fracture prognosis.| 0.802115772747805|
|   3|29416580|                            Evaluation of Risk Factors for Second Hip Fractures in Elderly Patients.|0.8011982022259172|
|   4|37068971|An analysis of cost and complications for patients sustaining a second, contralateral hip f

### 4.2 More clinical queries

Two of these deliberately **avoid the standard term**. *"bone healing failure"* never says *nonunion*, and
*"wrist fracture"* never says *distal radius*. These test whether the model has learned meaning rather than spelling.

In [15]:
DEMO_QUERIES = [
    "tibial shaft intramedullary nailing",
    "bone healing failure after fixation",
    "deep infection open fracture debridement antibiotics",
    "wrist fracture volar plate",
]

# Each test query returns its full list of 50 matching articles
for query in DEMO_QUERIES:
    print(f"=== {query}")
    engine.search(query).show(TOP_K, truncate=100)

=== tibial shaft intramedullary nailing
+----+--------+----------------------------------------------------------------------------------------------------+------------------+
|rank|    pmid|                                                                                               title|        similarity|
+----+--------+----------------------------------------------------------------------------------------------------+------------------+
|   1|34223791| Intramedullary Nailing of Paediatric Tibial Fractures: Comparison between Flexible and Rigid Nails.|0.8743257578421479|
|   2|27140219|Surgical factors contributing to nonunion in femoral shaft fracture following intramedullary nail...|0.8511134676408231|
|   3|22150351|                                          Tibial shaft fracture after removal of an ACE tibial nail.|0.8373636803396303|
|   4|40260178|              Use of controlled nail dynamization technique for femoral shaft hypertrophic nonunion.|0.7994306495773895|
|   5|20

### 4.3 Evaluation: precision@k against an independent label and a keyword baseline

**Relevance label.** PubMed librarians tag each indexed article with MeSH subject headings. An article counts as
*relevant* to a query if it carries one of the query's target headings. The tags are assigned by humans and neither
search method sees them, so neither method is marking its own work.

**Pool.** 2,000+ recent articles are not yet MeSH-indexed, which means they have no label. Both methods are scored on the
MeSH-indexed subset only.

**Baseline.** A keyword ranker using the same cleaned tokens. It ranks articles by the number of distinct query words
they contain, then by total query-word frequency.

In [16]:
@dataclass(frozen=True)
class EvalQuery:
    '''A test query and the MeSH headings that make an article relevant to it.'''

    text: str
    mesh_labels: tuple[str, ...]


# Test queries and the MeSH headings that count as relevant for each one
EVAL_QUERIES = [
    EvalQuery("hip fracture elderly mortality surgery delay", ("Hip Fractures", "Femoral Neck Fractures")),
    EvalQuery("tibial shaft intramedullary nailing", ("Tibial Fractures",)),
    EvalQuery("bone healing failure after fixation", ("Fractures, Ununited",)),
    EvalQuery("deep infection open fracture debridement antibiotics",
              ("Fractures, Open", "Surgical Wound Infection", "Osteomyelitis")),
    EvalQuery("wrist fracture volar plate", ("Radius Fractures", "Wrist Injuries")),
]

# Evaluation pool: MeSH-indexed articles only, with the "a;b;c" tag string split into an array
labelled = (
    corpus.filter(F.col("mesh").isNotNull() & (F.col("mesh") != ""))  # keep MeSH-indexed articles
    .withColumn("mesh_list", F.split("mesh", ";"))                    # "Hip Fractures;Aged" -> ["Hip Fractures", "Aged"]
    .cache()
)
labelled_pmids = [row.pmid for row in labelled.select("pmid").collect()]
# True/False per article, in the same order as the search index: True = part of the evaluation pool
eval_pool = np.isin(engine.index.pmids, labelled_pmids)
print(f"evaluation pool (MeSH-indexed articles): {len(labelled_pmids):,}")

evaluation pool (MeSH-indexed articles): 4,114


In [17]:
def relevant_pmids(labelled: DataFrame, mesh_labels: tuple[str, ...]) -> set[int]:
    '''PMIDs of the pool articles tagged with at least one of the target MeSH headings.'''
    targets = F.array(*[F.lit(label) for label in mesh_labels])  # the target headings as a Spark array
    # arrays_overlap = True when the article's tags and the targets share at least one heading
    return {row.pmid for row in labelled.filter(F.arrays_overlap("mesh_list", targets)).select("pmid").collect()}


def keyword_search(labelled: DataFrame, query_tokens: list[str], k: int = TOP_K) -> list[int]:
    '''
    Baseline without Word2Vec: rank by the number of distinct query words an article contains (overlap),
    then by how often those words appear (tf). Ties are broken by pmid so the result is reproducible.
    '''
    query_words = F.array(*[F.lit(token) for token in query_tokens])  # cleaned query words as a Spark array
    ranked = (
        # overlap: number of distinct query words present in the article
        labelled.withColumn("overlap", F.size(F.array_intersect("tokens_clean", query_words)))
        # tf: total occurrences of query words in the article (repeats count)
        .withColumn("tf", F.size(F.filter("tokens_clean", lambda t: F.array_contains(query_words, t))))
        .orderBy(F.desc("overlap"), F.desc("tf"), "pmid")
        .limit(k)
    )
    return [row.pmid for row in ranked.select("pmid").collect()]


def precision_at_k(ranked_pmids: list[int], relevant: set[int], k: int) -> float:
    '''Share of the first k results that are relevant. Example: 35 relevant in the top 50 -> 0.70.'''
    return sum(pmid in relevant for pmid in ranked_pmids[:k]) / k


def evaluate(query: EvalQuery, engine: SemanticSearch, labelled: DataFrame,
             pool: BoolArray) -> tuple[str, float, float, float, float, float]:
    '''Score both methods on one query: base rate, then precision@10 and @50 for keyword and Word2Vec.'''
    relevant = relevant_pmids(labelled, query.mesh_labels)  # the "answer key" for this query
    # Top 50 PMIDs from each method, both searching the same MeSH-indexed pool
    w2v = [row.pmid for row in engine.search(query.text, pool=pool).select("pmid").collect()]
    kw = keyword_search(labelled, engine.tokens(query.text))
    base_rate = len(relevant) / int(pool.sum())  # the precision a random ranking would get
    return (query.text, round(base_rate, 3),
            precision_at_k(kw, relevant, 10), precision_at_k(w2v, relevant, 10),
            precision_at_k(kw, relevant, 50), precision_at_k(w2v, relevant, 50))


# 1. One row per test query (kw = keyword baseline, w2v = Word2Vec search, p@10 / p@50 = precision at 10 / 50)
eval_df = spark.createDataFrame([evaluate(q, engine, labelled, eval_pool) for q in EVAL_QUERIES],
                                ["query", "base_rate", "kw_p@10", "w2v_p@10", "kw_p@50", "w2v_p@50"])
eval_df.show(truncate=False)

# 2. Average of every numeric column over the five queries
eval_df.select(*[F.round(F.avg(c), 3).alias(f"mean_{c}") for c in eval_df.columns[1:]]).show()

+----------------------------------------------------+---------+-------+--------+-------+--------+
|query                                               |base_rate|kw_p@10|w2v_p@10|kw_p@50|w2v_p@50|
+----------------------------------------------------+---------+-------+--------+-------+--------+
|hip fracture elderly mortality surgery delay        |0.184    |1.0    |1.0     |0.94   |0.96    |
|tibial shaft intramedullary nailing                 |0.106    |1.0    |0.8     |0.98   |0.82    |
|bone healing failure after fixation                 |0.102    |0.5    |0.6     |0.4    |0.7     |
|deep infection open fracture debridement antibiotics|0.128    |0.9    |0.9     |0.88   |0.9     |
|wrist fracture volar plate                          |0.158    |1.0    |1.0     |1.0    |1.0     |
+----------------------------------------------------+---------+-------+--------+-------+--------+

+--------------+------------+-------------+------------+-------------+
|mean_base_rate|mean_kw_p@10|mean_w2v

### 4.4 Findings

* **Both methods are far above chance.** About 14% of the pool is relevant to a typical query, while both rankers
  reach 84 to 88% precision@50.
* **Word2Vec wins when the wording differs.** For *"bone healing failure after fixation"*, which never uses the word
  *nonunion*, Word2Vec finds 35 nonunion papers in its top 50 and the keyword ranker finds 20 (0.70 vs 0.40). This is the case the
  ROI idea targets: a surgeon's phrasing does not have to match the authors' phrasing.
* **Keywords win on exact anatomical terms.** For *"tibial shaft intramedullary nailing"*, Word2Vec also pulls in
  *femoral* shaft nailing papers. They are close in meaning but not the requested bone (0.82 vs 0.98).
  Averaged word vectors blur specific anatomy.
* **Limitations.** Five queries are a small test set. MeSH is a coarse label, so a *hip fracture* paper about anaesthesia
  counts as relevant to a mortality query. Averaging word vectors also ignores word order.
  A practical tool would combine the two scores (hybrid ranking), using keywords for precise terms and Word2Vec for recall
  across different wording.

## 5. Train a model to predict: topic classification

The assignment asks to *"train a model to predict"*. Word2Vec itself is trained by prediction (§3). Here its output is
also used for an explicit **supervised prediction task**: given only an article's 100-dimensional Word2Vec vector,
predict which of the eight trauma topics it belongs to (hip fracture, nonunion, ankle, ...).

* **Features:** the averaged Word2Vec document vector (the same vectors the search uses).
* **Label:** `source_topic`, the PubMed query that retrieved the article.
* **Model:** multinomial logistic regression (Spark MLlib), trained on 80% of the articles and tested on the 20% it never saw.

If the vectors capture clinical meaning, a simple linear model should separate the topics well above the majority-class
baseline. Word2Vec never sees the labels, so the vectors are not tuned to this task.

In [18]:
def build_topic_dataset(model: Word2VecModel, corpus: DataFrame) -> DataFrame:
    '''One row per article: Word2Vec document vector (features) and numeric topic label (label).'''
    with_vectors = model.transform(corpus)  # adds the averaged Word2Vec vector of each article
    # StringIndexer maps each topic name to a number (0, 1, 2, ...), because Spark classifiers need numeric labels
    indexer = StringIndexer(inputCol="source_topic", outputCol="label").fit(with_vectors)
    return indexer.transform(with_vectors).select("pmid", "source_topic", "wordvectors", "label")


def train_topic_classifier(train: DataFrame) -> LogisticRegressionModel:
    '''Fit a multinomial logistic regression that predicts the topic from the document vector.'''
    classifier = LogisticRegression(
        featuresCol="wordvectors",  # input: the 100-dimensional document vector
        labelCol="label",           # target: the numeric topic
        maxIter=100,                # optimisation steps, enough to converge here
        family="multinomial",       # eight classes, not a yes/no problem
    )
    return classifier.fit(train)


def score(predictions: DataFrame, metric: str) -> float:
    '''Evaluate held-out predictions with one Spark metric ("accuracy" or weighted "f1").'''
    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName=metric)
    return evaluator.evaluate(predictions)


def majority_baseline(train: DataFrame, test: DataFrame) -> float:
    '''Accuracy of always predicting the most common training topic: the score to beat.'''
    majority_label = train.groupBy("label").count().orderBy(F.desc("count")).first()["label"]  # most frequent topic
    return test.filter(F.col("label") == majority_label).count() / test.count()


# 1. Features + labels, then a reproducible 80/20 split into training and test articles
topic_data = build_topic_dataset(model, corpus)
train_set, test_set = topic_data.randomSplit([1 - TEST_FRACTION, TEST_FRACTION], seed=SEED)

# 2. Train on the 80% only, then predict the topic of the 20% the model has never seen
classifier = train_topic_classifier(train_set)
predictions = classifier.transform(test_set)  # adds a "prediction" column

# 3. Report the held-out scores next to the baseline they must beat
print(f"train articles: {train_set.count():,} | test articles: {test_set.count():,}")
print(f"test accuracy:     {score(predictions, 'accuracy'):.3f}")
print(f"test weighted F1:  {score(predictions, 'f1'):.3f}")
print(f"majority baseline: {majority_baseline(train_set, test_set):.3f}")

train articles: 4,949 | test articles: 1,264
test accuracy:     0.851
test weighted F1:  0.850
majority baseline: 0.129


In [19]:
# Accuracy per topic on the held-out articles: which topics the vectors separate well, and which they confuse
is_correct = (F.col("label") == F.col("prediction")).cast("double")  # 1.0 if right, 0.0 if wrong; its mean = accuracy
(
    predictions.groupBy("source_topic")
    .agg(F.count("*").alias("test_articles"), F.round(F.avg(is_correct), 3).alias("accuracy"))
    .orderBy(F.desc("accuracy"))
    .show(truncate=False)
)

+------------------+-------------+--------+
|source_topic      |test_articles|accuracy|
+------------------+-------------+--------+
|distal_radius     |194          |0.969   |
|hip_fracture      |163          |0.926   |
|pelvic_acetabular |172          |0.901   |
|nonunion          |203          |0.877   |
|ankle_fracture    |173          |0.867   |
|fracture_infection|175          |0.76    |
|polytrauma        |128          |0.664   |
|tibial_nail       |56           |0.643   |
+------------------+-------------+--------+



### 5.1 Findings

* **The Word2Vec vectors predict the topic well.** The classifier gets **85.1%** accuracy on 1,264 held-out articles
  (weighted F1 0.85). Always guessing the most common topic would score 12.9%.
* **Errors follow clinical overlap.** Distal radius (97%) and hip fracture (93%) are almost always right. Tibial nailing (64%)
  and polytrauma (66%) are the weakest: tibial nailing papers often discuss nonunion, and polytrauma papers cover many
  injuries at once, so these topics genuinely overlap in content.
* This confirms quantitatively what §3.1 showed by example: the unsupervised Word2Vec model has learned clinically
  meaningful structure. That structure is what the search in §4 relies on.

In [20]:
# Shut down Spark and release its memory and CPU
spark.stop()